# Lab 1: Interactive Image Processing Application

In this lab, you will create an interactive image processing application using OpenCV, Matplotlib, and ipywidgets.

### Objectives
- Understand how to use OpenCV for image processing in Python.
- Learn to use ipywidgets for creating interactive applications.
- Familiarize with Matplotlib for displaying images and histograms.

### Prerequisites
- Basic knowledge of Python programming.
- Familiarity with image processing concepts.


In [ ]:
# Import necessary libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import os


## Setup
Create an output directory to save processed images.


In [ ]:
# Define the path for the outputs directory
output_path = './outputs/'
# Use os.makedirs to create the directory if it doesn't exist
os.makedirs(output_path, exist_ok=True)


## Global Variables
Initialize global variables to keep track of the images. These will be used across different callbacks.


In [ ]:
# Global variables
original_bgr = None
original_rgb = None
processed_rgb = None


## Output Areas
Create output areas for displaying messages, images, and histograms.


In [ ]:
# Create output widgets
msg_out = widgets.Output()
img_out = widgets.Output()
hist_out = widgets.Output()


## Define Controls
Define various controls needed for the interactive interface.


In [ ]:
# Create a button to browse images
browse_btn = widgets.Button(description='Browse Image')
# Create a button to convert images to grayscale
grayscale_btn = widgets.Button(description='Convert to Grayscale')
# Create a slider for brightness adjustment
brightness_slider = widgets.IntSlider(description='Brightness', min=-100, max=100, value=0)
# Create dropdowns for resizing and rotation
resize_dropdown = widgets.Dropdown(options=[('1.0', 1.0), ('0.75', 0.75), ('0.5', 0.5)], description='Resize')
rotation_dropdown = widgets.Dropdown(options=[('0', 0), ('90', 90), ('180', 180), ('270', 270)], description='Rotate')
# Create a button to plot histogram
hist_btn = widgets.Button(description='Plot Histogram')
# Create a button to save image
save_btn = widgets.Button(description='Save Image')


## Helper Functions
Define functions to show images and histograms.


In [ ]:
def show_images(original_rgb, processed_rgb):
    with img_out:
        img_out.clear_output(wait=True)
        fig, ax = plt.subplots(1, 2, figsize=(10, 5))
        ax[0].imshow(original_rgb)
        ax[0].set_title('Original Image')
        ax[0].axis('off')
        ax[1].imshow(processed_rgb)
        ax[1].set_title('Processed Image')
        ax[1].axis('off')
        plt.show()

def show_histogram(processed_rgb):
    with hist_out:
        hist_out.clear_output(wait=True)
        gray = cv2.cvtColor(processed_rgb, cv2.COLOR_RGB2GRAY)
        plt.figure(figsize=(5, 3))
        plt.hist(gray.ravel(), bins=256, range=(0, 256), color='black')
        plt.title('Histogram')
        plt.show()


## Callback Functions
Implement callback functions for each control.


In [ ]:
def on_browse_image_click(b):
    from IPython.display import clear_output
    from ipywidgets import FileUpload
    uploader = FileUpload(accept='.png, .jpg, .jpeg', multiple=False)
    display(uploader)
    def on_upload_change(change):
        global original_bgr, original_rgb, processed_rgb
        if uploader.value:
            for name, file_info in uploader.value.items():
                content = file_info['content']
                with open(name, 'wb') as f:
                    f.write(content)
                original_bgr = cv2.imread(name)
                original_rgb = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)
                processed_rgb = original_rgb.copy()
                show_images(original_rgb, processed_rgb)
                with msg_out:
                    msg_out.clear_output()
                    print(f'Loaded image: {name}')
        clear_output()
    uploader.observe(on_upload_change, names='value')

def on_grayscale_click(b):
    global processed_rgb
    gray = cv2.cvtColor(original_rgb, cv2.COLOR_RGB2GRAY)
    processed_rgb = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    show_images(original_rgb, processed_rgb)

def on_brightness_change(change):
    global processed_rgb
    brightness_value = change['new']
    processed_rgb = cv2.convertScaleAbs(original_rgb, alpha=1, beta=brightness_value)
    show_images(original_rgb, processed_rgb)

def on_resize_change(change):
    global processed_rgb
    scale = change['new']
    width = int(original_rgb.shape[1] * scale)
    height = int(original_rgb.shape[0] * scale)
    processed_rgb = cv2.resize(original_rgb, (width, height))
    show_images(original_rgb, processed_rgb)
    with msg_out:
        msg_out.clear_output()
        print(f'Resized to: {processed_rgb.shape[1]}x{processed_rgb.shape[0]}')

def on_rotation_change(change):
    global processed_rgb
    angle = change['new']
    if angle == 90:
        processed_rgb = cv2.rotate(original_rgb, cv2.ROTATE_90_CLOCKWISE)
    elif angle == 180:
        processed_rgb = cv2.rotate(original_rgb, cv2.ROTATE_180)
    elif angle == 270:
        processed_rgb = cv2.rotate(original_rgb, cv2.ROTATE_90_COUNTERCLOCKWISE)
    else:
        processed_rgb = original_rgb.copy()
    show_images(original_rgb, processed_rgb)

def on_plot_histogram_click(b):
    show_histogram(processed_rgb)

def on_save_image_click(b):
    output_filename = os.path.join(output_path, 'processed_image.png')
    cv2.imwrite(output_filename, cv2.cvtColor(processed_rgb, cv2.COLOR_RGB2BGR))
    with msg_out:
        msg_out.clear_output()
        print(f'Saved image to: {output_filename}')


## Connect Callbacks
Connect the defined functions to widgets as callbacks.


In [ ]:
# Connect the buttons to their callbacks
browse_btn.on_click(on_browse_image_click)
grayscale_btn.on_click(on_grayscale_click)
brightness_slider.observe(on_brightness_change, names='value')
resize_dropdown.observe(on_resize_change, names='value')
rotation_dropdown.observe(on_rotation_change, names='value')
hist_btn.on_click(on_plot_histogram_click)
save_btn.on_click(on_save_image_click)


## Layout
Arrange the widgets in a user-friendly layout.


In [ ]:
# Display the layout
controls = widgets.VBox([browse_btn, grayscale_btn, brightness_slider, resize_dropdown, rotation_dropdown, hist_btn, save_btn])
right_side = widgets.VBox([img_out, hist_out])
ui = widgets.HBox([controls, right_side])
display(msg_out, ui)
with msg_out:
    print('Please click Browse Image to start.')
